In [ ]:
import numpy as np
import csv
import pandas as pd
from collections import defaultdict

import matplotlib.pyplot as plt
#%matplotlib qt

with open("IMDB top 1000.csv", 'r', encoding='utf-8') as f: #file_name 불필요 (수정_wk)
    reader = csv.reader(f)
    next(reader)
    data_list = [row for row in reader]

data_array = np.array(data_list)

data_title = data_array[:,1:2]
data_genre = data_array[:,4]
data_rate = data_array[:,5:6]
released_year = np.array([year[0][-5:-1] for year in data_title])
data_ry = released_year.reshape(-1,1) #array로 변경
#2-1 
data_cast = data_array[:,8:9]
cast_spr = np.char.split(data_cast[:,0],' | ') # spr : Separate
# Cast_Director 분리
data_drc = np.array([drc_row[0].replace('Director: ','').replace('Directors: ','').strip() for drc_row in cast_spr])
data_drc = data_drc.reshape(-1,1)
#print(data_drc)
#2-2
#Cast_Star 분리
data_star = np.array([star_row[1].replace('Stars: ','').strip() for star_row in cast_spr])
data_star = data_star.reshape(-1,1)
#print(data_star)
#3 
data_stack = np.column_stack((data_title, data_genre, data_rate,data_ry,data_drc,data_star))
column_header_step1 = "Title,Genre,Rate,Released_Year,Director,Star"
data_stack_array = np.array(data_stack)

#4 
fil_NaN = np.all(data_stack_array != '', axis=1)
data_filter = data_stack_array[fil_NaN]
data_step1 = data_filter
print("\n=== Step 1: 데이터준비 ===")
print(data_step1)
print("-"*60)
df.info()
print("-"*60)
# Step 2 -------------------------------------------------------------------------------------
# 전체 영화 개수, 평균 평점, 최고 평점, 최저 평점 출력
print("\n=== Step 2: 기본적인 데이터 탐색 ===")

#rt_idx : Rate index(평점이 있는 열의 위치 번호라는 의미)
#astype은 float로 바꾸라는 코드. 기존 str이라 float로 바꾸어야 계산이 가능.
rate_idx = 2 #Rate index
data_rate = data_step1[:, rate_idx].astype(float)

# 전체 영화 개수
data_mov_count = data_step1.shape[0]

# 평균, 최고, 최저 평점 계산
# avg는 값이 8.0975로 나와서 Round 사용하여 8.1로 계산되게끔 함.
data_rate_avg = round(data_rate.mean(), 1)
data_rate_max = data_rate.max()
data_rate_min = data_rate.min()

#출력
print(f"총 영화 개수: {data_mov_count}")
print(f"평점 평균: {data_rate_avg}")
print(f"최고 평점: {data_rate_max}")
print(f"최저 평점: {data_rate_min}")
print("-"*60)

# Step 3 -------------------------------------------------------------------------------------
print("\n=== Step 3: 평점이 높은 영화 찾기 ===")
# : 평점이 높은 영화 찾기

data_rate_max = data_rate.max()

# np.where를 통해 "조건을 만족하는 데이터의 위치" 찾기 가능
top_idx = np.where(data_rate == data_rate_max)[0]

print("최고 평점 영화 목록:")

for i in top_idx:
    title = data_step1[i, 0]
    rate = data_step1[i, 2]

#출력
print(f"{title} - 평점: {rate}")
print("-"*60)
# Step 4 -------------------------------------------------------------------------------------
print("\n=== Step 4: 특정 장르별 평균 평점 분석 ===")
#4-1 필요한 열 추출 및 타입 지정
#data_rte = data_step1[:,2].astype(float) # 위의 data_rate 사용

#4-2 장르 분리 및 리스트에 모으기
genre_all_lst = []
#for gnr_row in data_gnr:
for genre_row in data_genre:
    genre_spr = [genre_itm.strip() for genre_itm in genre_row.split(",")]
    genre_all_lst.extend(genre_spr)

#4-3 장르 중복 제거
data_genre_all = np.array(genre_all_lst, dtype=object)
data_genre_uni = np.unique(data_genre_all)

#4-4 장르별 평균 평점 계산
genre_avg_lst = []
for genre_itm in data_genre_uni:
    data_genre_flt = (np.char.find(data_genre,genre_itm) >= 0)
    data_rate_avg = data_rate[data_genre_flt].mean()
    genre_avg_lst.append((genre_itm, data_rate_avg))
data_genre_avg = np.array(genre_avg_lst, dtype=object)
print(data_genre_avg)
#4-5 평균 평점 기준으로 정렬
data_rate_avg_val = data_genre_avg[:,1].astype(float)
data_rate_avg_ord = np.argsort(data_rate_avg_val)[::-1]
data_genre_avg_srt = data_genre_avg[data_rate_avg_ord]

#출력
#for genre_itm, rate_avg in data_genre_avg_srt:
    #print(f"{genre_itm}: {float(rate_avg):.2f}")
print("-"*60)
# Step 5: 장르별 평균 평점 분석
print("\n=== Step 5: 년도별 평균 평점 ===")

ry_idx = 3     #Releasd_year index (ry)
rt_idx = 2     #Rate index (rt) 

#연도와 평점을 숫자 타입으로 변환 과정
years = data_step1[:, ry_idx].astype(int)
rates = data_step1[:, rt_idx].astype(float)

#연도 고유값 추출(중복 년도 재거, 오름차순 표시)
unique_years = np.unique(years)

year_avg = []

for y in unique_years:
    #해당 연도의 평점만 표시
    flt = years == y
    avg_rate = rates[flt].mean()

    year_avg.append([y, avg_rate])

#출력
for y, avg in year_avg:
    print(f"{int(y)}년 average rating: {round(avg, 2)}")



#해당방식으로 저장하면 엑셀 파일에서 년도와 평균 평점만 확인가능
# import csv
# np.savetxt(
#     "IMDB_Top1000_Year_Avg_Rate.csv",
#     year_avg,
#     delimiter=",",
#     fmt=["%d", "%.2f"],
#     header="Released_Year,Avg_Rate",
#     comments="",
#     encoding="utf-8"
# )

# Step 6: 시각화 --------------------------------------------------------------
print("\n=== Step 6: 시각화 ===")

years_num = unique_years
years_avg = np.array([row[1] for row in year_avg])

plt.figure(figsize=(10, 5))
plt.plot(years_num, years_avg, marker='o')
plt.title('Yearly Rate_Average')
plt.xlabel('year')
plt.ylabel('rate_average')
plt.grid(True)
plt.xticks(years_num[::5], rotation=45)  # 5년 간격
plt.tight_layout()
plt.show()
# Step 7: 연도별 장르영화갯수 추이 ----------------------------------------------------
print("\n=== Step 7: 연도별 장르영화갯수 추이 ===")

# 장르별로 각 연도에서 영화 개수 세기 (threshold 5로 낮춤)
year_genre_count = defaultdict(lambda: defaultdict(int))
threshold = 5  # "주요 장르 없음" 너무 많아서 기준 낮춤

for i in range(len(data_step1)):
    year = int(data_step1[i, ry_idx])
    genres = [g.strip() for g in str(data_step1[i, 1]).split(',')]
    
    for genre in genres:
        year_genre_count[year][genre] += 1

# 연도별 출력 (5개 이상 장르만)
print(f"연도별 장르별 영화 수 (기준: {threshold}편 이상)")
print("-" * 60)
recent_years = sorted([y for y in year_genre_count.keys() if y >= 1920])  # 1920년 이후 
for year in recent_years:
    major_genres = {g: count for g, count in year_genre_count[year].items() if count >= threshold}
    if major_genres:
        top_genres = sorted(major_genres.items(), key=lambda x: x[1], reverse=True)[:3]
        print(f"{year:4d}년: {', '.join([f'{g}({count})' for g,count in top_genres])}")
print("-" * 60)

# 전체 통계
print("전체 기간 Top 10 장르:")
genre_total = defaultdict(int)
for year_data in year_genre_count.values():
    for genre, count in year_data.items():
        genre_total[genre] += count

for genre, total in sorted(genre_total.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"  {genre:12s}: {total:3d}편")
print("-" * 60)


# Step 7-1: 연도별 장르영화갯수 추이 시각화 --------------------------------------------
print("\n=== Step 7-1: 연도별 장르영화갯수 추이 시각화 ===")

# 1. 상위 10개 장르 선정 및 확인
top_n = 10
top_genres = sorted(genre_total.items(), key=lambda x: x[1], reverse=True)[:top_n]
top_genre_names = [genre for genre, _ in top_genres]

print(f"상위 {top_n}개 장르:")
for i, (genre, total) in enumerate(top_genres, 1):
    print(f"  {i}. {genre:12s}: {total:4d}편")
print()

# 2. 연도 데이터 준비 (2000년 이후)
years_plot = sorted([y for y in year_genre_count.keys() if 1920 <= y <= 2020])
genre_year_data = {genre: [] for genre in top_genre_names}

for year in years_plot:
    for genre in top_genre_names:
        count = year_genre_count[year].get(genre, 0)
        genre_year_data[genre].append(count)

# 3. 시각화
plt.figure(figsize=(15, 12))

# 상단: 선 그래프 (상위 10개 장르 추이)
plt.subplot(2, 1, 1)
colors = plt.cm.tab10(np.linspace(0, 1, top_n))  # 10개 색상 팔레트
for i, genre in enumerate(top_genre_names):
    plt.plot(years_plot, genre_year_data[genre], marker='o', linewidth=2.5, 
             label=f'{genre}({top_genres[i][1]}편)', color=colors[i], 
             markersize=5, alpha=0.85)

plt.title(f'Top 1000 Yearly genre_count(1920-2020)', fontsize=15, fontweight='bold', pad=20)
plt.xlabel('year')
plt.ylabel('count_movie')
plt.legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize=10)
plt.grid(True, alpha=0.3)
plt.xticks(years_plot[::5], rotation=45)


plt.show()




print("-" * 60)
# Step 8: 연도별 최고평점 영화 정보 -----------------------------------------------
print("\n=== Step 8: 연도별 최고평점 영화 정보 ===")

year_best_movies = {}
max_yearly_rate = {}  # 연도별 최고 평점 저장

for year in unique_years:
    year_mask = years == year
    year_rates = rates[year_mask]
    if len(year_rates) == 0:
        continue
    
    best_rate = year_rates.max()
    max_yearly_rate[year] = best_rate
    
    best_indices = np.where((years == year) & (rates == best_rate))[0]
    
    for idx in best_indices[:2]:  # 연도당 최대 2개
        title = str(data_step1[idx, 0])[:50]
        genre = str(data_step1[idx, 1])[:30]
        rate = float(data_step1[idx, 2])  # 명시적 float 변환
        
        if year not in year_best_movies:
            year_best_movies[year] = []
        year_best_movies[year].append({
            'title': title,
            'genre': genre,
            'rate': rate
        })

# 테이블 출력 (numpy.str_ 문제 해결)
print("연도 | 타이틀" + " "*(38-len("타이틀")) + "| 장르" + " "*(20-len("장르")) + "| 평점")
print("-" * 80)
recent_years_with_movies = sorted([y for y in year_best_movies.keys() if y >= 1920])
for year in recent_years_with_movies[:]:  # 전체
    for movie in year_best_movies[year]:
        print(f"{int(year):4d} | {movie['title']:50.50} | {movie['genre']:25.25} | {movie['rate']:.1f}")
print("-" * 80)


# Step 9: 연도별 TOP1 장르와 최고평점 영화 장르 일치 여부 분석
print("\n=== Step 9: 연도별 TOP1 장르와 최고평점 영화 장르 일치 여부 ===")

# 기존에 이미 import된 모듈만 사용 (pandas, defaultdict 추가 import 불필요)
ry_idx = 3  # Released_year index  
genre_idx = 1  # Genre index

# 1. 연도별 TOP1 장르 찾기
year_top_genre = {}
for year in unique_years:
    year_mask = years == year
    year_movies = data_step1[year_mask]
    
    if len(year_movies) == 0:
        continue
    
    # 해당 연도의 장르 카운트
    year_genre_count = defaultdict(int)
    for i in range(len(year_movies)):
        genres = [g.strip() for g in str(year_movies[i, genre_idx]).split(',')]
        for genre in genres:
            year_genre_count[genre] += 1
    
    # TOP1 장르
    if year_genre_count:
        top_genre = max(year_genre_count.items(), key=lambda x: x[1])[0]
        year_top_genre[year] = top_genre

# 2. 연도별 일치 여부 확인 및 표 출력
print("연도 | TOP1장르    | 최고평점영화 장르      | 일치")
print("-" * 60)

match_results = []
recent_years_with_movies = sorted([y for y in year_best_movies.keys() if y >= 1920])

for year in recent_years_with_movies:
    if year not in year_top_genre:
        continue
    
    top_genre = year_top_genre[year]
    best_movies = year_best_movies[year]
    best_movie_genres = set()
    
    # 최고평점 영화 장르들 수집
    for movie in best_movies:
        genres = [g.strip() for g in movie['genre'].split(',')]
        best_movie_genres.update(genres)
    
    # 일치 여부 확인
    is_match = 1 if top_genre in best_movie_genres else 0
    match_symbol = '●' if is_match else '○'
    
    match_results.append([int(year), top_genre, ', '.join(best_movie_genres), is_match])
    
    # 표 출력
    print(f"{int(year):4d} | {top_genre:10s} | {', '.join(best_movie_genres)[:30]:30s} | {match_symbol}")

print("-" * 60)

# 3. 통계
total_years = len(match_results)
match_count = sum(row[3] for row in match_results)
match_rate = round((match_count / total_years) * 100, 1) if total_years > 0 else 0

print(f"총 분석 연도 수: {total_years}개")
print(f"일치한 연도 수: {match_count}개") 
print(f"일치율: {match_rate}%")


